# 02 - New Orleans Data Preparation

This notebook creates the city-level working subset used by the rest of the academic project.

The raw Yelp review and user files are very large. To keep the project manageable, we stream through the JSONL files and write only records relevant to New Orleans.

## Outputs

This notebook creates:

```text
data/interim/new_orleans/businesses.jsonl
data/interim/new_orleans/reviews.jsonl
data/interim/new_orleans/users.jsonl
data/interim/new_orleans/summary.json
```

The generated files are ignored by Git because they are derived data artifacts.

In [1]:
from pathlib import Path
import csv
import json
from collections import Counter
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_YELP_DIR = DATA_DIR / "raw" / "yelp"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

BUSINESS_PATH = RAW_YELP_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = RAW_YELP_DIR / "yelp_academic_dataset_review.json"
USER_PATH = RAW_YELP_DIR / "yelp_academic_dataset_user.json"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp


In [2]:
TARGET_CITY = "New Orleans"
TARGET_STATE = "LA"
DATASET_SLUG = "new_orleans"

CITY_INTERIM_DIR = INTERIM_DIR / DATASET_SLUG
CITY_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

BUSINESSES_OUTPUT = CITY_INTERIM_DIR / "businesses.jsonl"
REVIEWS_OUTPUT = CITY_INTERIM_DIR / "reviews.jsonl"
USERS_OUTPUT = CITY_INTERIM_DIR / "users.jsonl"
SUMMARY_OUTPUT = CITY_INTERIM_DIR / "summary.json"

print(CITY_INTERIM_DIR)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\interim\new_orleans


In [3]:
def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} at line {line_number}") from exc


def write_jsonl(records, path):
    count = 0
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False))
            file.write("\n")
            count += 1
    return count

In [4]:
selected_business_ids = set()

def selected_business_records():
    for record in iter_jsonl(BUSINESS_PATH):
        city = (record.get("city") or "").strip().casefold()
        state = (record.get("state") or "").strip().upper()
        if city == TARGET_CITY.casefold() and state == TARGET_STATE:
            selected_business_ids.add(record["business_id"])
            yield record

business_count = write_jsonl(selected_business_records(), BUSINESSES_OUTPUT)
print(f"Selected businesses: {business_count:,}")

Selected businesses: 6,215


In [5]:
selected_user_ids = set()
min_review_date = None
max_review_date = None

def selected_review_records():
    global min_review_date, max_review_date
    for record in iter_jsonl(REVIEW_PATH):
        if record.get("business_id") in selected_business_ids:
            selected_user_ids.add(record["user_id"])
            review_date = record.get("date")
            if review_date:
                min_review_date = review_date if min_review_date is None else min(min_review_date, review_date)
                max_review_date = review_date if max_review_date is None else max(max_review_date, review_date)
            yield record

review_count = write_jsonl(selected_review_records(), REVIEWS_OUTPUT)
print(f"Selected reviews: {review_count:,}")
print(f"Unique reviewing users: {len(selected_user_ids):,}")
print(f"Date range: {min_review_date} to {max_review_date}")

Selected reviews: 635,521
Unique reviewing users: 245,421
Date range: 2005-03-14 18:07:51 to 2022-01-19 19:47:59


In [6]:
def selected_user_records():
    for record in iter_jsonl(USER_PATH):
        if record.get("user_id") in selected_user_ids:
            yield record

user_count = write_jsonl(selected_user_records(), USERS_OUTPUT)
print(f"Selected user profiles: {user_count:,}")

Selected user profiles: 245,419


In [7]:
summary = {
    "target_city": TARGET_CITY,
    "target_state": TARGET_STATE,
    "dataset_slug": DATASET_SLUG,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "business_count": business_count,
    "review_count": review_count,
    "unique_reviewing_user_count": len(selected_user_ids),
    "matched_user_profile_count": user_count,
    "min_review_date": min_review_date,
    "max_review_date": max_review_date,
    "outputs": {
        "businesses": str(BUSINESSES_OUTPUT),
        "reviews": str(REVIEWS_OUTPUT),
        "users": str(USERS_OUTPUT),
    },
}

with SUMMARY_OUTPUT.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)
    file.write("\n")

summary

{'business_count': 6215,
 'created_at_utc': '2026-05-12T17:30:52.351011+00:00',
 'dataset_slug': 'new_orleans',
 'matched_user_profile_count': 245419,
 'max_review_date': '2022-01-19 19:47:59',
 'min_review_date': '2005-03-14 18:07:51',
 'outputs': {'businesses': 'C:\\Users\\mehdi\\OneDrive\\Documents\\community-forecasting-yelp\\data\\interim\\new_orleans\\businesses.jsonl',
             'reviews': 'C:\\Users\\mehdi\\OneDrive\\Documents\\community-forecasting-yelp\\data\\interim\\new_orleans\\reviews.jsonl',
             'users': 'C:\\Users\\mehdi\\OneDrive\\Documents\\community-forecasting-yelp\\data\\interim\\new_orleans\\users.jsonl'},
 'review_count': 635521,
 'target_city': 'New Orleans',
 'target_state': 'LA',
 'unique_reviewing_user_count': 245421}

## Preparation Result From Current Run

The first completed extraction produced:

- **6,215** New Orleans businesses,
- **635,521** reviews,
- **245,421** unique reviewing users,
- **245,419** matched user profiles,
- review dates from **2005-03-14** to **2022-01-19**.

Two reviewing users did not have matched profiles in the user table. This is small relative to the subset size and can be documented as a minor data consistency issue.